In [30]:
CONFIG = {
    "experiment": "notebook_paths",
    "display_simulation": True,
    "visualization": {
        "playback_speed": 1.0,
        "max_fps": 60,
        "ground_height_m": -1.0,
    },
    "ik": {
        # Use "nimble" for differentiable IK or "opensim" for legacy IK.
        "backend": "nimble",
        
        # Brace wrist motion and forearm pronation/supination.
        "lock_wrist": True,
        "locked_wrist_degrees": {
            # KINARM grip: palm/opening faces upward.
            "pro_sup": -30.0,
            "deviation": 0.0,
            "flexion": 0.0,
        },
        "coordinate_regularization_weight": 0.001,
        "nimble": {
            "wsl_distribution": "Ubuntu",
            "python": "/home/braydenk/.venvs/motor-meta-nimble/bin/python",
            "iterations": 12,
            "damping": 1e-5,
            "posture_weight": 1e-5,
        },
    },
    
    "proprioception": {
        # May be absolute or relative to the repository. Set to None to derive
        # the checkpoint directory from model_family and the two seeds.
        "model_path": "trained_models/experiment_causal_flag-pcr_optimized_linear_extended_5_5_letter_reconstruction_joints/spatiotemporal_4_8-8-32-64_7171_0_9",
        "model_family": "extended",
        "coefficient_seed": 0,
        "training_seed": 9,
        # Change to "zeros" for exact checkpoint-training padding or
        # "reflect" for reflected temporal boundaries.
        "temporal_padding_mode": "replicate",
    },
    
    "muscle_lengths": {
        # Match the four-coordinate muscle inputs used to train the checkpoint.
        "drive_distal_coordinates": False,
    },
    
    "path": {
        "generator": "process/generate_paths/generatereachpath.py ", #draw.py or generatereachpath.py 
        "sample_rate_hz": 240,
        "reach_cm": 10.0,
        "max_displacement_cm": 45.0,
        "workspace_ellipse": {
            "center_cm": [-10.0, 0.0],
            "radii_cm": [30.0, 40.0],
        },
        "path_speed_cm_s": 20.0,
        "return_to_rest": False,
        "rest_pose_degrees": {
            "elv_angle": 20.0,
            "shoulder_elv": 40.0,
            "shoulder_rot": 25.0,
            "elbow_flexion": 85.0,
        },
        "timing_seconds": {
            "hold_before": 1.65,
            "reach": 0.50,
            "hold_target": 0.50,
            "return": 0.50,
            "hold_after": 1.65,
        },
    },
}

In [31]:
# Materialize CONFIG as YAML and prepare the shared runner.
# The generated file is under ignored outputs/.
from pathlib import Path
import yaml
from run_pipeline import Pipeline, STAGE_NAMES

CONFIG_PATH = Path("outputs/notebook_config.yaml").resolve()
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
with CONFIG_PATH.open("w", encoding="utf-8") as stream:
    yaml.safe_dump(CONFIG, stream, sort_keys=False)

pipeline = Pipeline(CONFIG_PATH)
print(f"Configuration: {CONFIG_PATH}")
print(f"Outputs:       {pipeline.output_dir}")
print(f"Stages:        {', '.join(STAGE_NAMES)}")

Configuration: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\outputs\notebook_config.yaml
Outputs:       C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\outputs\notebook_paths
Stages:        path, inverse-kinematics, motion, muscle-lengths, muscle-signals, inference


## 1. Generate paths
This opens the drawing window when the draw generator is selected above.

In [32]:
pipeline.run_stage("path")


=== path: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\generate_paths\generatereachpath.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\generate_paths\\generatereachpath.py'], returncode=0)

## 2. Inverse kinematics

In [33]:
pipeline.run_stage("inverse-kinematics")


=== inverse-kinematics: Nimble Physics (WSL) ===


CompletedProcess(args=['wsl', '-d', 'Ubuntu', '--', 'env', 'MOTOR_META_EXPERIMENT=notebook_paths', 'MOTOR_META_CONFIG=/mnt/c/Users/brayk/OneDrive/Documents/CMU-Classes/PhiLab/MotorMetamersuPNC/outputs/notebook_config.yaml', '/home/braydenk/.venvs/motor-meta-nimble/bin/python', '/mnt/c/Users/brayk/OneDrive/Documents/CMU-Classes/PhiLab/MotorMetamersuPNC/process/nimble_ik.py'], returncode=0)

## 3. Generate OpenSim motion

In [34]:
pipeline.run_stage("motion")


=== motion: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\gencenterout.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\gencenterout.py'], returncode=0)

## Review motion
Optional. This launches the same OpenSim playback/control UI used by the command-line pipeline.

In [35]:
pipeline.display_simulation()


=== motion-review: OpenSim visualizer ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\utils\\display_sim.py'], returncode=0)

## 4. Extract muscle lengths

In [36]:
pipeline.run_stage("muscle-lengths")


=== muscle-lengths: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\extractcenterout.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\extractcenterout.py'], returncode=0)

## 5. Compute spindle signals

In [37]:
pipeline.run_stage("muscle-signals")


=== muscle-signals: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\computefrcenterout.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\computefrcenterout.py'], returncode=0)

## 6. Run neural inference
This requires the pretrained checkpoint described in the README.

In [38]:
pipeline.run_stage("inference")


=== inference: C:\Users\brayk\OneDrive\Documents\CMU-Classes\PhiLab\MotorMetamersuPNC\process\centeroutinference.py ===


CompletedProcess(args=['c:\\Users\\brayk\\anaconda3\\envs\\motor-meta-simplified\\python.exe', 'C:\\Users\\brayk\\OneDrive\\Documents\\CMU-Classes\\PhiLab\\MotorMetamersuPNC\\process\\centeroutinference.py'], returncode=0)

## 7. Visualize proprioception
Select any inferred trajectory. The strong red line is the wrist path predicted from spindle activity; the translucent blue line is the original wrist path supplied to the pipeline. Replay draws both paths together in experiment time.

In [40]:
import json
import uuid
import pandas as pd
from IPython.display import HTML, display

prediction_dir = pipeline.output_dir / "predictions"
prediction_files = sorted(
    path for path in prediction_dir.glob("*.csv") if path.name != "summary.csv"
)
if not prediction_files:
    raise FileNotFoundError(
        f"No inference CSVs found in {prediction_dir}. Run Block 6 first."
    )

trajectories = {}
for csv_path in prediction_files:
    frame = pd.read_csv(csv_path)
    trajectories[csv_path.stem] = {
        "time": frame["time_s"].tolist(),
        "true": frame[["true_wrist_X_cm", "true_wrist_Y_cm"]].values.tolist(),
        "pred": frame[["pred_wrist_X_cm", "pred_wrist_Y_cm"]].values.tolist(),
    }

viewer_id = f"proprioception-{uuid.uuid4().hex}"
payload = json.dumps(trajectories)
display(HTML(f'''
<div id="{viewer_id}" style="font-family:system-ui;max-width:850px">
  <div style="display:flex;gap:12px;align-items:center;margin:8px 0">
    <label>Trajectory <select class="trajectory"></select></label>
    <button class="replay" style="padding:5px 14px">Restart</button>
    <button class="pause" style="padding:5px 14px">Play</button>
    <button class="previous" title="Previous frame">-1 frame</button>
    <button class="next" title="Next frame">+1 frame</button>
    <label>Speed <select class="speed"><option value="1">1x</option><option value="2">2x</option><option value="4">4x</option></select></label>
    <span class="clock" style="font-variant-numeric:tabular-nums"></span>
  </div>
  <input class="frame" type="range" min="1" value="1" step="1" style="width:100%;margin-bottom:8px">
  <canvas width="820" height="560" style="width:100%;border:1px solid #ddd;background:white"></canvas>
  <div style="display:flex;gap:22px;margin-top:5px">
    <span style="color:#c0392b">━ Predicted from proprioception</span>
    <span style="color:rgba(31,119,180,.55)">━ Original path</span>
  </div>
</div>
<script>
(() => {{
  const data = {payload};
  const root = document.getElementById('{viewer_id}');
  const select = root.querySelector('.trajectory');
  const speed = root.querySelector('.speed');
  const frame = root.querySelector('.frame');
  const pause = root.querySelector('.pause');
  const clock = root.querySelector('.clock');
  const canvas = root.querySelector('canvas');
  const ctx = canvas.getContext('2d');
  let animation = null, currentFrame = 1, playing = false, startedAt = 0, startedFrame = 1;
  Object.keys(data).forEach(name => {{ const option=document.createElement('option'); option.value=name; option.textContent=name; select.appendChild(option); }});

  function geometry(item) {{
    const points = item.true.concat(item.pred);
    let xs=points.map(p=>p[0]), ys=points.map(p=>p[1]);
    let xmin=Math.min(...xs), xmax=Math.max(...xs), ymin=Math.min(...ys), ymax=Math.max(...ys);
    let span=Math.max(xmax-xmin, ymax-ymin, 1);
    let cx=(xmin+xmax)/2, cy=(ymin+ymax)/2, pad=55, scale=(Math.min(canvas.width,canvas.height)-2*pad)/span;
    return {{map:p=>[canvas.width/2+(p[0]-cx)*scale, canvas.height/2-(p[1]-cy)*scale], xmin, xmax, ymin, ymax, pad}};
  }}
  function line(points, count, map, color, width) {{
    if (!count) return; ctx.beginPath(); let p=map(points[0]); ctx.moveTo(...p);
    for(let i=1;i<count;i++) {{ p=map(points[i]); ctx.lineTo(...p); }}
    ctx.strokeStyle=color; ctx.lineWidth=width; ctx.lineJoin='round'; ctx.lineCap='round'; ctx.stroke();
    p=map(points[count-1]); ctx.beginPath(); ctx.arc(p[0],p[1],4,0,2*Math.PI); ctx.fillStyle=color; ctx.fill();
  }}
  function axes(g) {{
    ctx.font='12px system-ui'; ctx.lineWidth=1; ctx.textAlign='center'; ctx.textBaseline='top';
    for(let i=0;i<=5;i++) {{
      const x=g.xmin+(g.xmax-g.xmin)*i/5, px=g.map([x,g.ymin])[0];
      ctx.beginPath(); ctx.moveTo(px,g.pad); ctx.lineTo(px,canvas.height-g.pad); ctx.strokeStyle='#eeeeee'; ctx.stroke();
      ctx.fillStyle='#555'; ctx.fillText(x.toFixed(1),px,canvas.height-g.pad+7);
    }}
    ctx.textAlign='right'; ctx.textBaseline='middle';
    for(let i=0;i<=5;i++) {{
      const y=g.ymin+(g.ymax-g.ymin)*i/5, py=g.map([g.xmin,y])[1];
      ctx.beginPath(); ctx.moveTo(g.pad,py); ctx.lineTo(canvas.width-g.pad,py); ctx.strokeStyle='#eeeeee'; ctx.stroke();
      ctx.fillStyle='#555'; ctx.fillText(y.toFixed(1),g.pad-7,py);
    }}
    ctx.strokeStyle='#999'; ctx.strokeRect(g.pad,g.pad,canvas.width-2*g.pad,canvas.height-2*g.pad);
  }}
  function draw(count) {{
    const item=data[select.value], g=geometry(item), n=Math.max(1,Math.min(count,item.time.length));
    currentFrame=n; frame.max=item.time.length; frame.value=n;
    ctx.clearRect(0,0,canvas.width,canvas.height);
    axes(g);
    ctx.fillStyle='#555'; ctx.font='13px system-ui'; ctx.fillText('Wrist X (cm)', canvas.width/2-35, canvas.height-14);
    ctx.save(); ctx.translate(16,canvas.height/2+30); ctx.rotate(-Math.PI/2); ctx.fillText('Wrist Y (cm)',0,0); ctx.restore();
    line(item.true,n,g.map,'rgba(31,119,180,.35)',5);
    line(item.pred,n,g.map,'#c0392b',3);
    clock.textContent = n ? `${{item.time[n-1].toFixed(2)}} / ${{item.time[item.time.length-1].toFixed(2)}} s` : '';
  }}
  function stop() {{ if(animation) cancelAnimationFrame(animation); animation=null; playing=false; pause.textContent='Play'; }}
  function showComplete() {{ stop(); draw(data[select.value].time.length); }}
  function replay() {{
    stop(); draw(1); play();
  }}
  function play() {{
    if(playing) {{ stop(); return; }}
    const item=data[select.value]; if(currentFrame>=item.time.length) draw(1);
    playing=true; pause.textContent='Pause'; startedAt=performance.now(); startedFrame=currentFrame;
    function step(now) {{
      const elapsed=(now-startedAt)/1000*Number(speed.value);
      const target=item.time[startedFrame-1]+elapsed; let index=startedFrame-1;
      while(index<item.time.length && item.time[index]<=target) index++; draw(Math.max(1,index));
      if(index<item.time.length && playing) animation=requestAnimationFrame(step); else stop();
    }}
    animation=requestAnimationFrame(step);
  }}
  function seek(delta) {{ stop(); draw(currentFrame+delta); }}
  select.addEventListener('change',showComplete);
  root.querySelector('.replay').addEventListener('click',replay); pause.addEventListener('click',play);
  root.querySelector('.previous').addEventListener('click',()=>seek(-1));
  root.querySelector('.next').addEventListener('click',()=>seek(1));
  frame.addEventListener('input',()=>{{ stop(); draw(Number(frame.value)); }});
  showComplete();
}})();
</script>
'''))


Some individual components, including the Nimble IK operation and neural network, are differentiable. This stage-oriented notebook currently saves and reloads artifacts between processes, so it does **not** preserve an end-to-end computation graph and cannot yet backpropagate a loss to the input path.